# Non-CSS surface-code decoding

This notebook decodes the full, non-CSS GKP surface-code lattice. It mirrors the CSS-sector example, but keeps the coupled `q` and `p` quadratures together.

In [29]:
using Random
using LinearAlgebra
using LatticeDecoder

Random.seed!(2);

function is_not_logical_error(logical_check, residual; atol = 1e-5)
    logical_coordinates = logical_check' * residual
    return all(abs(x - round(x)) < atol for x in logical_coordinates)
end;



Build a distance-3 surface code and derive the full parity-check matrix `H` and correction generator `G` in the `qqpp` convention.

In [30]:
d = 3

code = GKP_Surface_Code(d, false);
M = code.code;
J = code.J;

H = -M * J;
G = J * inv(M);
logical_check = inv(M);

logicals = code.logical;


Draw one full displacement vector and decode it with serial belief propagation.

In [31]:
noise_std = 0.5 / sqrt(2 * pi)
max_iter = size(H, 2)
decoder = "lsd";

error_vector = sample_error(noise_std, size(H, 2));
received = copy(error_vector);

tanner_graph = initialize_tanner_graph(H);
bp_estimate = run_serial_belief_propagation!(
    tanner_graph,
    received,
    noise_std,
    max_iter,
    decoder;
);

decoded_integer_correction = hard_decision(bp_estimate, H);


correction = received - G * decoded_integer_correction;
residual = error_vector - correction;

# println(logical_check' * residual)

println("BP correct: ", is_not_logical_error(logical_check, residual))
println("BP + X correct: ", is_not_logical_error(logical_check, residual + logicals[1, :]))
println("BP + Z correct: ", is_not_logical_error(logical_check, residual + logicals[2, :]))
println("BP + Y correct: ", is_not_logical_error(logical_check, residual + logicals[1, :] + logicals[2, :]))


any_correct = is_not_logical_error(logical_check, residual) ||
    is_not_logical_error(logical_check, residual + logicals[1, :]) ||
    is_not_logical_error(logical_check, residual + logicals[2, :]) ||
    is_not_logical_error(logical_check, residual + logicals[1, :] + logicals[2, :])

println("Any correct: ", any_correct)


BP correct: true
BP + X correct: false
BP + Z correct: false
BP + Y correct: false
Any correct: true


# Small distance sweep


In [34]:
function surface_code_noncss_decode(d::Int)
    code = GKP_Surface_Code(d, false);
    M = code.code;
    J = code.J;

    H = -M * J;
    G = J * inv(M);
    logical_check = inv(M);

    return H, G, logical_check
end

function surface_code_noncss_decode_fails(H, G, logical_check, noise_std; decoder = "nearest", search_interval = 1.0)
    error_vector = sample_error(noise_std, size(H, 2))
    received = copy(error_vector)
    tanner_graph = initialize_tanner_graph(H)

    bp_estimate = run_serial_belief_propagation!(
        tanner_graph,
        received,
        noise_std,
        size(H, 2),
        decoder;
        search_interval = search_interval,
    )

    decoded_integer_correction = hard_decision(bp_estimate, H)
    correction = received - G * decoded_integer_correction
    residual = error_vector - correction

    return !is_not_logical_error(logical_check, residual)
end

surface_code_noncss_decode_fails (generic function with 1 method)

In [ ]:
distances = [3, 5, 7, 9]
sigmas = collect(range(0.3, 0.8; length = 6)) ./ sqrt(2 * pi)
samples_per_point = 25_000

failure_rates = Dict{Int, Vector{Float64}}()

for distance in distances
    H_d, G_d, L_d = surface_code_noncss_decode(distance)
    rates = Float64[]

    for sigma in sigmas
        failures = count(_ -> surface_code_noncss_decode_fails(H_d, G_d, L_d, sigma), 1:samples_per_point)
        push!(rates, failures / samples_per_point)
    end

    failure_rates[distance] = rates
end

for d in distances
    println("Distance $d failure rates: ", failure_rates[d])
end

In [ ]:
p = plot(
    xlabel = "noise std σ",
    ylabel = "logical failure rate",
    title = "CSS surface-code decoding ($(basis) sector)",
    legend = :topleft,
    yscale = :log10,
)

for distance in distances
    plot!(p, sigmas * sqrt(2 * pi), failure_rates[distance]; marker = :circle, label = "d = $(distance)")
end
p